# 🏥 Medical Insurance Cost Prediction
### Final Data Science Project: End-to-End Analysis & Machine Learning

**Author:** Ayush Verma | BCA, Chandigarh University | Data Science & Analytics

---
## 1. Problem Statement
Health insurers must estimate how much a person is likely to cost them so that premiums are fair and sustainable. 
This project analyses a real-world medical insurance dataset to:
1. Understand **which factors drive medical charges** (EDA & visualisation)
2. Build and **compare machine learning models** that predict an individual's charges
3. **Tune and validate** the best model and explain *why* it predicts what it does
4. Package the result as a **portfolio-ready** project with a reusable prediction function

## 2. Project Workflow
`Load data → Clean → EDA → Feature engineering → Preprocessing pipeline → Model comparison (cross-validation) → Hyperparameter tuning → Final evaluation → Explainability → Save model → Conclusions`

## 3. Dataset
**Medical Cost Personal Dataset** (public, widely used): 1,338 records with `age`, `sex`, `bmi`, `children`, `smoker`, `region` and the target **`charges`** (yearly medical cost billed by insurance, in USD).

## 4. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.titleweight": "bold"})
RANDOM_STATE = 42

## 5. Load the Data

In [ ]:
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
try:
    df = pd.read_csv("data/insurance.csv")   # local copy (GitHub repo)
except FileNotFoundError:
    df = pd.read_csv(URL)                     # online copy (Colab)
print("Shape:", df.shape)
df.head()

## 6. Data Cleaning & Overview

In [ ]:
df.info()
print("\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)
df.describe().round(2)

## 7. Exploratory Data Analysis

### 7.1 Distribution of the target (charges)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["charges"], bins=40, kde=True, color="#e76f51", ax=axes[0])
axes[0].set_title("Distribution of Charges")
sns.histplot(np.log1p(df["charges"]), bins=40, kde=True, color="#2a9d8f", ax=axes[1])
axes[1].set_title("Distribution of log(Charges)")
plt.tight_layout(); plt.show()
print(f"💡 Insight: Charges are heavily right-skewed (mean ${df.charges.mean():,.0f} vs median ${df.charges.median():,.0f}). A few very expensive patients pull the average up.")

### 7.2 Smoking: the biggest driver

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x="smoker", y="charges", hue="smoker", palette={"no": "#2a9d8f", "yes": "#e76f51"}, legend=False, ax=axes[0])
axes[0].set_title("Charges: Smokers vs Non-smokers")
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", palette={"no": "#2a9d8f", "yes": "#e76f51"}, alpha=0.7, ax=axes[1])
axes[1].set_title("Age vs Charges (coloured by smoker)")
plt.tight_layout(); plt.show()
med = df.groupby("smoker")["charges"].median()
print(f"💡 Insight: Median charges are ${med['yes']:,.0f} for smokers vs ${med['no']:,.0f} for non-smokers, about {med['yes']/med['no']:.1f}x higher.")

### 7.3 Age and BMI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", palette={"no": "#2a9d8f", "yes": "#e76f51"}, alpha=0.7, ax=axes[0])
axes[0].axvline(30, color="grey", linestyle="--"); axes[0].set_title("BMI vs Charges (dashed line = BMI 30, obesity threshold)")
sns.regplot(data=df[df.smoker == "no"], x="age", y="charges", scatter_kws={"alpha": 0.4}, line_kws={"color": "red"}, ax=axes[1])
axes[1].set_title("Age vs Charges (non-smokers only)")
plt.tight_layout(); plt.show()
print("💡 Insight: Charges rise steadily with age. High BMI (>30) only becomes a major cost driver for SMOKERS, which suggests an interaction effect.")

### 7.4 Sex, children and region

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["sex", "children", "region"]):
    m = df.groupby(col)["charges"].mean()
    sns.barplot(x=m.index.astype(str), y=m.values, hue=m.index.astype(str), palette="viridis", legend=False, ax=ax)
    ax.set_title(f"Average Charges by {col}"); ax.set_ylabel("Average Charges ($)"); ax.set_xlabel("")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():,.0f}", (p.get_x() + p.get_width()/2, p.get_height()), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print("💡 Insight: Sex, number of children and region show only small differences compared with smoking and age.")

### 7.5 Correlation

In [ ]:
num = df.copy()
num["smoker_flag"] = (num["smoker"] == "yes").astype(int)
sns.heatmap(num[["age", "bmi", "children", "smoker_flag", "charges"]].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap"); plt.show()
print(f"💡 Insight: Smoking has by far the strongest correlation with charges (r = {num['smoker_flag'].corr(num['charges']):.2f}).")

## 8. Feature Engineering
Domain knowledge and the EDA above suggest two useful features:
- **`obese`**: BMI ≥ 30 (a medical risk threshold)
- **`smoker_obese`**: smoker AND obese (the interaction seen in the BMI scatter plot)

In [ ]:
data = df.copy()
data["obese"] = (data["bmi"] >= 30).astype(int)
data["smoker_obese"] = ((data["smoker"] == "yes") & (data["obese"] == 1)).astype(int)
print(data.groupby(["smoker", "obese"])["charges"].median().round(0).unstack())

## 9. Preprocessing Pipeline & Train/Test Split
A scikit-learn `Pipeline` + `ColumnTransformer` keeps preprocessing and the model together, which prevents data leakage and makes the final model easy to save and reuse.

In [ ]:
X = data.drop("charges", axis=1)
y = data["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]
bin_cols = ["obese", "smoker_obese"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 10. Model Comparison (5-fold Cross-Validation)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, m in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", m)])
    r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    mae = -cross_val_score(pipe, X_train, y_train, cv=kf, scoring="neg_mean_absolute_error")
    rows.append({"Model": name, "CV R² (mean)": r2.mean().round(3), "CV R² (std)": r2.std().round(3), "CV MAE ($)": mae.mean().round(0)})
cv_results = pd.DataFrame(rows).sort_values("CV R² (mean)", ascending=False).reset_index(drop=True)
cv_results

In [ ]:
cv_results.set_index("Model")["CV R² (mean)"].sort_values().plot(kind="barh", color="#2a9d8f", figsize=(8, 4))
plt.title("Cross-Validated R² by Model"); plt.xlabel("R²"); plt.xlim(0, 1); plt.show()

## 11. Hyperparameter Tuning (Gradient Boosting)

In [ ]:
pipe = Pipeline([("prep", preprocess), ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))])
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
}
grid = GridSearchCV(pipe, param_grid, cv=kf, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)
print("Best parameters:", grid.best_params_)
print(f"Best CV R²: {grid.best_score_:.3f}")

## 12. Final Evaluation on the Unseen Test Set

In [ ]:
best_model = grid.best_estimator_
pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f"MAE  : ${mae:,.0f}")
print(f"RMSE : ${rmse:,.0f}")
print(f"R²   : {r2:.3f}")

# Baseline for comparison: always predicting the average
baseline_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"\nBaseline (predict the mean) MAE: ${baseline_mae:,.0f}  ->  model reduces error by {1 - mae/baseline_mae:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, pred, alpha=0.6, color="#2a9d8f")
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual Charges ($)"); axes[0].set_ylabel("Predicted Charges ($)")
axes[0].set_title("Actual vs Predicted"); axes[0].legend()

resid = y_test - pred
sns.histplot(resid, bins=30, kde=True, color="#e76f51", ax=axes[1])
axes[1].axvline(0, color="black", linestyle="--"); axes[1].set_title("Residuals (Actual - Predicted)")
plt.tight_layout(); plt.show()

## 13. Explainability: What drives the predictions?

In [ ]:
feat_names = best_model.named_steps["prep"].get_feature_names_out()
feat_names = [f.split("__")[-1] for f in feat_names]
imp = pd.Series(best_model.named_steps["model"].feature_importances_, index=feat_names).sort_values()
imp.plot(kind="barh", color="#2a9d8f", figsize=(8, 5))
plt.title("Feature Importance (Gradient Boosting)"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()
top3 = imp.sort_values(ascending=False).head(3)
print("💡 Insight: Top drivers:", ", ".join(f"{k} ({v:.0%})" for k, v in top3.items()))

## 14. Save the Model & Make Predictions

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/insurance_model.joblib")
print("Model saved ✅")

def predict_charges(age, sex, bmi, children, smoker, region):
    """Predict yearly medical insurance charges for one person."""
    obese = int(bmi >= 30)
    row = pd.DataFrame([{
        "age": age, "sex": sex, "bmi": bmi, "children": children,
        "smoker": smoker, "region": region,
        "obese": obese, "smoker_obese": int(smoker == "yes" and obese == 1),
    }])
    return float(best_model.predict(row)[0])

print(f"Non-smoker, 30 yrs, BMI 24 : ${predict_charges(30, 'male', 24, 0, 'no', 'southwest'):,.0f}")
print(f"Smoker,     30 yrs, BMI 24 : ${predict_charges(30, 'male', 24, 0, 'yes', 'southwest'):,.0f}")
print(f"Smoker,     30 yrs, BMI 35 : ${predict_charges(30, 'male', 35, 0, 'yes', 'southwest'):,.0f}")

## 15. Conclusions

| # | Finding |
|---|---------|
| 1 | **Smoking** is the single biggest driver of charges (smokers cost several times more) |
| 2 | **Age** raises charges steadily |
| 3 | **High BMI** matters mostly for smokers, an interaction captured by the engineered `smoker_obese` feature |
| 4 | Sex, children and region have little impact |
| 5 | After tuning, **Gradient Boosting** scores marginally best (CV R² ≈ 0.85), but a simple **Linear Regression** is almost as good once the engineered features are added, so a simple model is a strong choice here. Test R² ≈ 0.90, and error is ~75% lower than the "predict the mean" baseline |

## 16. Business Recommendations
- Use smoking status, age and BMI as the core risk factors in premium models.
- Offer **smoking-cessation and weight-management programmes**. They target the most expensive group.
- Use the model to **flag high-cost profiles early** for care-management programmes.

## 17. Limitations & Future Work
- Only 1,338 records and 6 features: real insurers use far richer data (medical history, claims).
- Charges are right-skewed. Trying a log-transform of the target or other models (XGBoost, LightGBM) could help.
- Results show association, not causation, and the data may not represent every population.
- Future: build a **Streamlit web app** for live predictions and add SHAP-based explanations.

**Tools:** Python • Pandas • NumPy • Matplotlib • Seaborn • Scikit-learn • Joblib

*Author: Ayush Verma*